# Explore cấu hình model NeMo ASR (CPU)

Soi cấu hình các model ASR, chạy **CPU**. Logic in nằm ở `utils/inspect_helpers.py` (đã test).

**An toàn cho máy (đọc trước khi chạy):**
- Cell setup **giới hạn 4 nhân CPU** — tránh torch pin cả 8 nhân làm treo máy (đã gặp: load average vọt 17 trên 8 nhân).
- **Mỗi model load xong được giải phóng ngay** (`del m; ih.free()`), không giữ 3 model 0.6B cùng lúc trong RAM.
- Model `parakeet` / `nemotron` mỗi cái ~2.4GB, cache ở `~/.cache/huggingface`.

## 0. Setup (giới hạn nhân CPU TRƯỚC khi import)

In [ ]:
import os
# Dat TRUOC khi import torch/nemo de co hieu luc -> tranh pin het 8 nhan, treo may
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'

import sys
sys.path.insert(0, os.path.abspath('.'))  # de import package utils khi chay tu thu muc notebooks/
from utils import inspect_helpers as ih
import pandas as pd

ih.set_cpu_threads(4)  # chot lai gioi han nhan cho torch

SMALL    = 'nvidia/stt_en_conformer_ctc_small'         # ~13M, decoder CTC, tai nhanh
PARAKEET = 'nvidia/parakeet-tdt-0.6b-v2'               # ~618M, decoder TDT
NEMOTRON = 'nvidia/nemotron-speech-streaming-en-0.6b'  # ~600M, cache-aware streaming

rows = []  # gom dong tom tat (nhe) de so sanh, KHONG giu model

## 1. Conformer-CTC small (baseline nhỏ)

Làm quen định dạng output với model nhẹ.

In [ ]:
m = ih.load(SMALL)
ih.print_overview(m, SMALL)
rows.append(ih.summary_row(m, SMALL))
del m; ih.free()  # giai phong ngay

## 2. Parakeet-TDT-0.6b-v2 (decoder TDT)

Chú ý dòng `TDT: joint_out = vocab + 1 blank + N duration` — đặc trưng TDT (dự đoán token + thời lượng).

In [ ]:
m = ih.load(PARAKEET)
ih.print_overview(m, PARAKEET)
rows.append(ih.summary_row(m, PARAKEET))
del m; ih.free()

## 3. Nemotron-Speech-Streaming-0.6b (cache-aware, streaming)

Encoder cache-aware FastConformer + decoder RNNT. Tiếng Anh.

In [ ]:
m = ih.load(NEMOTRON)
ih.print_overview(m, NEMOTRON)
rows.append(ih.summary_row(m, NEMOTRON))
del m; ih.free()

## 4. Bảng so sánh (dựng từ các dòng tóm tắt, không giữ model)

In [ ]:
pd.DataFrame(rows)

## 5. Xem cấu hình chi tiết (YAML) một model

Load lại đúng một model để soi config, rồi giải phóng. Đổi `target` và `section` ('encoder'/'decoder'/'joint'/'decoding').

In [ ]:
target = PARAKEET
m = ih.load(target)
print('### encoder ###')
print(ih.config_yaml(m, 'encoder'))
print('### decoding ###')
print(ih.config_yaml(m, 'decoding'))
del m; ih.free()